In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Step 2 — Extract All Zip Files

CNN models need images inside folders, not zip files.
So we will unzip them into a training dataset folder.

In [3]:
import os
import shutil

BASE_PATH = "/content"
# Create fresh directory structure
os.makedirs(os.path.join(BASE_PATH, "Datasets"), exist_ok=True)

print("Fresh directory structure created ")

Fresh directory structure created 


In [4]:
!cp "/content/drive/MyDrive/Major Project/Datasets Zipped/AI.zip" /content/
!cp "/content/drive/MyDrive/Major Project/Datasets Zipped/CGI.zip" /content/
!cp "/content/drive/MyDrive/Major Project/Datasets Zipped/Edited.zip" /content/
!cp "/content/drive/MyDrive/Major Project/Datasets Zipped/Real.zip" /content/

In [5]:
!rm -rf /content/Datasets
!mkdir -p /content/Datasets/AI
!mkdir -p /content/Datasets/CGI
!mkdir -p /content/Datasets/Edited
!mkdir -p /content/Datasets/Real

!unzip -q -j AI.zip -d /content/Datasets/AI
!unzip -q -j CGI.zip -d /content/Datasets/CGI
!unzip -q -j Edited.zip -d /content/Datasets/Edited
!unzip -q -j Real.zip -d /content/Datasets/Real

In [6]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense,GlobalAveragePooling2D,Dropout,BatchNormalization,Activation
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras import regularizers
from tensorflow.keras.applications.resnet50 import ResNet50,preprocess_input

from sklearn.metrics import confusion_matrix,classification_report

In [7]:
!cp -r "/content/drive/MyDrive/Major Project/Test Datasets" /content/

In [8]:
train_path = "/content/Datasets"
test_path = "/content/Test Datasets"

In [9]:

train_datagen=ImageDataGenerator(
    preprocessing_function=preprocess_input,
    horizontal_flip=True,
    rotation_range=10,
    zoom_range=0.1,
    validation_split=0.2,
)

train_generator=train_datagen.flow_from_directory(
    train_path,
    target_size=(224,224),
    batch_size=64,
    class_mode="categorical",
    subset="training"
)

val_generator=train_datagen.flow_from_directory(
    train_path,
    target_size=(224,224),
    batch_size=64,
    class_mode="categorical",
    subset="validation"
)

Found 51815 images belonging to 4 classes.
Found 12952 images belonging to 4 classes.


In [10]:
test_datagen=ImageDataGenerator(preprocessing_function=preprocess_input)
test_generator=test_datagen.flow_from_directory(
    test_path,
    target_size=(224,224),
    batch_size=64,
    class_mode="categorical",
    shuffle=False
)

Found 4353 images belonging to 4 classes.


Step 6 — Load ResNet50 (Transfer Learning)

In [11]:
base_model=ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Freeze pretrained layers:

In [12]:
base_model.trainable = True
for layer in base_model.layers[:-10]:
    layer.trainable=False

Step 7 — Custom Classification Layers

In [13]:
x=base_model.output
x=GlobalAveragePooling2D()(x)
x=Dense(256,kernel_regularizer=regularizers.l2(1e-4))(x)
x=BatchNormalization()(x)
x=Activation("relu")(x)
x=Dropout(0.4)(x)
predictions=Dense(4,activation="softmax")(x)
model=Model(inputs=base_model.input,outputs=predictions)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 24,114,308 (91.99 MB)

 Trainable params: 4,991,748 (19.04 MB)

 Non-trainable params: 19,122,560 (72.95 MB)

In [14]:
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss=CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]

)

In [15]:
checkpoint=ModelCheckpoint(

    "/content/drive/MyDrive/Major Project/Rishabh CNN/best_resnet_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1

)

In [16]:
from tensorflow.keras.callbacks import EarlyStopping

EarlyStop=EarlyStopping(
    monitor="val_accuracy",
    patience=5,
    mode="max",
    verbose=1
)

Step 10 — Train Model

In [ ]:
history=model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=[checkpoint,EarlyStop],
    verbose=1

)

Epoch 1/10
810/810 ━━━━━━━━━━━━━━━━━━━━ 0s 863ms/step - accuracy: 0.7045 - loss: 0.9784
Epoch 1: val_accuracy improved from None to 0.84690, saving model to /content/drive/MyDrive/Major Project/Rishabh CNN/best_resnet_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Major Project/Rishabh CNN/best_resnet_model.keras
810/810 ━━━━━━━━━━━━━━━━━━━━ 903s 1s/step - accuracy: 0.7769 - loss: 0.8376 - val_accuracy: 0.8469 - val_loss: 0.7107
Epoch 2/10
810/810 ━━━━━━━━━━━━━━━━━━━━ 0s 847ms/step - accuracy: 0.8678 - loss: 0.6743
Epoch 2: val_accuracy improved from 0.84690 to 0.87739, saving model to /content/drive/MyDrive/Major Project/Rishabh CNN/best_resnet_model.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Major Project/Rishabh CNN/best_resnet_model.keras
810/810 ━━━━━━━━━━━━━━━━━━━━ 859s 1s/step - accuracy: 0.8655 - loss: 0.6717 - val_accuracy: 0.8774 - val_loss: 0.6678
Epoch 3/10
810/810 ━━━━━━━━━━━━━━━━━━━━ 0s 838ms/step - accuracy: 0.8952 - loss: 0.61

In [ ]:
best_model=tf.keras.models.load_model("/content/drive/MyDrive/Major Project/Rishabh CNN/best_resnet_model.keras")

In [ ]:
test_loss,test_acc=best_model.evaluate(test_generator)

print("Test Accuracy:",test_acc)

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


69/69 ━━━━━━━━━━━━━━━━━━━━ 34s 362ms/step - accuracy: 0.6258 - loss: 1.0333
Test Accuracy: 0.6257753372192383


In [ ]:
predictions=best_model.predict(test_generator)

y_pred=np.argmax(predictions,axis=1)

y_true=test_generator.classes

In [ ]:
cm=confusion_matrix(y_true,y_pred)

plt.figure(figsize=(8,6))

sns.heatmap(cm,annot=True,fmt="d",cmap="Blues")

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()

In [ ]:
class_names=list(test_generator.class_indices.keys())

print(classification_report(y_true,y_pred,target_names=class_names))

              precision    recall  f1-score   support

          AI       0.38      0.76      0.51       536
         CGI       0.98      0.67      0.80      2096
      Edited       0.50      0.86      0.63       921
        Real       0.46      0.14      0.22       800

    accuracy                           0.63      4353
   macro avg       0.58      0.61      0.54      4353
weighted avg       0.71      0.63      0.62      4353

